In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

bronze_table = f"{catalog}.f1_bronze.api_responses"

bronze_df = spark.table(bronze_table)

# Get one raw JSON sample so Spark can infer its nested structure.
sample_json = (
    bronze_df
    .orderBy(F.col("ingested_at").desc())
    .select("payload_json")
    .first()["payload_json"]
)

inferred_schema = (
    spark.range(1)
    .select(
        F.schema_of_json(F.lit(sample_json)).alias("json_schema")
    )
    .first()["json_schema"]
)

print(inferred_schema)

In [0]:
parsed_bronze_df = bronze_df.withColumn(
    "json_data",
    F.from_json(
        F.col("payload_json"),
        inferred_schema
    )
)

display(
    parsed_bronze_df.select("json_data")
)

In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

bronze_table = f"{catalog}.f1_bronze.api_responses"

bronze_df = spark.table(bronze_table)

# Use the newest raw API response.
bronze_df = (
    bronze_df
    .orderBy(F.col("ingested_at").desc())
    .limit(1)
)

# Read one JSON sample so Spark can understand its structure.
sample_json = bronze_df.select("payload_json").first()["payload_json"]

inferred_schema = (
    spark.range(1)
    .select(
        F.schema_of_json(F.lit(sample_json)).alias("json_schema")
    )
    .first()["json_schema"]
)

# Convert the JSON text into a structured Spark object.
parsed_bronze_df = bronze_df.withColumn(
    "json_data",
    F.from_json(
        F.col("payload_json"),
        inferred_schema
    )
)

print("Bronze JSON parsed successfully.")

In [0]:
races_df = (
    parsed_bronze_df
    .select(
        F.explode(
            "json_data.MRData.RaceTable.Races"
        ).alias("race")
    )
)

display(
    races_df.select(
        F.col("race.raceName").alias("race_name"),
        F.col("race.season").alias("season"),
        F.col("race.round").alias("round"),
        F.col("race.Circuit.circuitName").alias("circuit_name")
    )
)

In [0]:
results_exploded_df = (
    races_df
    .select(
        F.col("race.season").cast("int").alias("season"),
        F.col("race.round").cast("int").alias("round"),
        F.col("race.raceName").alias("race_name"),
        F.col("race.Circuit.circuitId").alias("circuit_id"),
        F.explode("race.Results").alias("result")
    )
)

In [0]:
silver_results_df = (
    results_exploded_df
    .select(
        F.concat_ws(
            "_",
            F.col("season").cast("string"),
            F.lpad(F.col("round").cast("string"), 2, "0")
        ).alias("race_id"),

        "season",
        "round",
        "race_name",
        "circuit_id",

        F.col("result.Driver.driverId").alias("driver_id"),
        F.col("result.Driver.code").alias("driver_code"),
        F.col("result.Driver.givenName").alias("driver_first_name"),
        F.col("result.Driver.familyName").alias("driver_last_name"),

        F.col("result.Constructor.constructorId").alias("constructor_id"),
        F.col("result.Constructor.name").alias("constructor_name"),

        F.col("result.grid").cast("int").alias("grid"),
        F.col("result.position").cast("int").alias("finish_position"),
        F.col("result.positionText").alias("finish_status"),
        F.col("result.points").cast("double").alias("points"),
        F.col("result.laps").cast("int").alias("laps_completed"),
        F.col("result.status").alias("race_status")
    )
)

display(
    silver_results_df.orderBy("finish_position")
)

In [0]:
from pyspark.sql import functions as F

# 1. Check for duplicate driver results.
duplicate_results_df = (
    silver_results_df
    .groupBy("race_id", "driver_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate keys found:", duplicate_results_df.count())
display(duplicate_results_df)

In [0]:
# 2. Check for missing important values.
missing_values_df = silver_results_df.filter(
    F.col("race_id").isNull()
    | F.col("driver_id").isNull()
    | F.col("constructor_id").isNull()
)

print("Rows with missing required values:", missing_values_df.count())
display(missing_values_df)

In [0]:
# 3. Check for invalid finishing positions.
invalid_positions_df = silver_results_df.filter(
    F.col("finish_position").isNull()
    | (F.col("finish_position") <= 0)
    | (F.col("finish_position") > 30)
)

print("Invalid finishing positions:", invalid_positions_df.count())
display(invalid_positions_df)

In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

silver_table = f"{catalog}.f1_silver.race_results"

silver_results_to_save_df = (
    silver_results_df
    .withColumn("loaded_at", F.current_timestamp())
)

(
    silver_results_to_save_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(f"Saved Silver table: {silver_table}")

In [0]:
saved_silver_df = spark.table(silver_table)

print("Silver row count:", saved_silver_df.count())

display(
    saved_silver_df
    .select(
        "race_id",
        "driver_code",
        "driver_last_name",
        "constructor_name",
        "finish_position",
        "points"
    )
    .orderBy("finish_position")
)